# Matrix multiplication, inversion and photo filters
**Linear Algebra mini project** — pipeline: real-world data (image) → matrix representation → linear transformation (filter) → inverse / rank / conditioning → application output.

**How to run on Colab:** Runtime → Run all. (Optional: run the upload cell to use your own photo; otherwise a synthetic test image is used.)

| File | Owner |
|---|---|
| `utils.py`, `main.py` | Person 1 |
| `filters.py` | Person 2 |
| `inverse.py` | Person 3 |
| report / math | Person 4 |

## 1. Project files (each cell writes one `.py` file — edit your own cell)

In [ ]:
%%writefile utils.py
"""
utils.py  --  PERSON 1: image <-> matrix pipeline (everyone imports this file)

Conventions (do not change without telling the team):
  * An image is a float64 NumPy array of shape (H, W, 3), values in [0, 1].
  * A filter is a 3x3 matrix A that acts on a COLUMN vector [R, G, B]^T:
        new_pixel = A @ old_pixel
    so "apply A first, then B" is the single matrix  B @ A.
"""
import os
import numpy as np
from PIL import Image


# ----------------------------------------------------------------------
# Core pipeline
# ----------------------------------------------------------------------
def load_image(path):
    """Read an image file -> float array (H, W, 3) in [0, 1].

    Concept: a colour image IS a matrix of numbers. Each pixel holds an
    RGB vector; we scale 0-255 down to 0-1 so we can do real arithmetic.
    """
    img = Image.open(path).convert("RGB")           # force 3 channels
    return np.asarray(img, dtype=np.float64) / 255.0


def apply_filter(img, A, clip=True):
    """Apply a 3x3 colour filter A to every pixel of img.

    Concept: stack all H*W pixels as columns of one 3 x (H*W) matrix P.
    Then ONE matrix product  A @ P  filters every pixel at once, and we
    reshape the result back into an image.

    clip=True  -> clamp to [0, 1] so the result is a valid image
    clip=False -> keep raw values (needed to test exact inverses)
    """
    A = np.asarray(A, dtype=np.float64)
    if A.shape != (3, 3):
        raise ValueError(f"Filter must be 3x3, got {A.shape}")
    if img.ndim != 3 or img.shape[2] != 3:
        raise ValueError(f"Image must be (H, W, 3), got {img.shape}")

    h, w, _ = img.shape
    pixels = img.reshape(-1, 3).T          # 3 x (H*W): one column per pixel
    new_pixels = A @ pixels                # the linear transformation
    out = new_pixels.T.reshape(h, w, 3)    # back to image shape
    return np.clip(out, 0.0, 1.0) if clip else out


def save_image(img, path):
    """Save a float (H, W, 3) image in [0, 1] as an 8-bit file. Creates folders."""
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)
    arr = np.rint(np.clip(img, 0.0, 1.0) * 255).astype(np.uint8)
    Image.fromarray(arr).save(path)


# ----------------------------------------------------------------------
# Shared helpers (used by Persons 2, 3, 4)
# ----------------------------------------------------------------------
def error_between(a, b):
    """Return (MSE, max absolute error) between two images."""
    d = np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64)
    return float(np.mean(d ** 2)), float(np.max(np.abs(d)))


def image_info(img):
    """Short text summary of an image array."""
    h, w, c = img.shape
    return (f"shape = {h} x {w} x {c}  ({h * w:,} pixels), "
            f"dtype = {img.dtype}, range = [{img.min():.3f}, {img.max():.3f}]")


def banner(stage, concept, purpose):
    """Print a stage header in the Concept -> Purpose -> Outcome viva format."""
    line = "=" * 72
    print(f"\n{line}\n{stage}\n  Concept : {concept}\n  Purpose : {purpose}\n{line}")


def make_sample_image(path, h=240, w=360):
    """Create a colourful synthetic test image so the code runs anywhere.
    Replace with a REAL photo (images/photo.jpg) for the final submission."""
    y, x = np.mgrid[0:h, 0:w]
    img = np.stack([x / w, y / h, 1 - (x + y) / (w + h)], axis=-1)
    for cx, cy, r, col in [(90, 90, 50, (0.95, 0.75, 0.1)),
                           (250, 80, 40, (0.8, 0.1, 0.2)),
                           (180, 170, 55, (0.1, 0.6, 0.3))]:
        mask = (x - cx) ** 2 + (y - cy) ** 2 < r ** 2
        img[mask] = col
    save_image(img, path)
    return path


def show_side_by_side(images, titles, save_to=None, show=True):
    """Display (and optionally save) several images in one row."""
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4))
    axes = np.atleast_1d(axes)
    for ax, im, t in zip(axes, images, titles):
        ax.imshow(np.clip(im, 0, 1))
        ax.set_title(t, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    if save_to:
        os.makedirs(os.path.dirname(save_to) or ".", exist_ok=True)
        fig.savefig(save_to, dpi=120)
    if show:
        plt.show()
    plt.close(fig)

In [ ]:
%%writefile filters.py
"""
filters.py -- PERSON 2
Filter matrices, filtered image saving, and composition test.
Keeps run(img, out_dir) -> dict for main.py and inverse.py.
"""

import numpy as np
from utils import (
    apply_filter,
    save_image,
    error_between,
    show_side_by_side,
)

# All filters act on column RGB vectors: new_pixel = A @ old_pixel
FILTERS = {
    "identity": np.eye(3),

    "grayscale": np.array([
        [1/3, 1/3, 1/3],
        [1/3, 1/3, 1/3],
        [1/3, 1/3, 1/3],
    ], dtype=float),

    "sepia": np.array([
        [0.393, 0.769, 0.189],
        [0.349, 0.686, 0.168],
        [0.272, 0.534, 0.131],
    ]),

    "swap_rb": np.array([
        [0, 0, 1],
        [0, 1, 0],
        [1, 0, 0],
    ], dtype=float),

    "saturate": np.diag([1.2, 0.75, 2.0]),

    "mix": np.array([
        [0.70, 0.15, 0.15],
        [0.15, 0.70, 0.15],
        [0.15, 0.15, 0.70],
    ]),

    # Brightness scaling: increase all RGB channels by 10%
    "brightness": 1.10 * np.eye(3),
}


def run(img, out_dir):
    """Apply and save each filter, then test matrix composition."""

    results = {}

    # Part 1: Apply each individual filter
    print("\nApplying individual filters:")

    for name, A in FILTERS.items():
        out = apply_filter(img, A, clip=True)
        save_image(out, f"{out_dir}/filtered_{name}.png")
        results[name] = out
        print(f"  Saved filtered_{name}.png")

    print(f"Applied {len(results)} filters: {', '.join(results)}")

    # Part 2: Composition test
    # A = red/blue swap; B = saturation
    A = FILTERS["swap_rb"]
    B = FILTERS["saturate"]

    print("\nMATRIX COMPOSITION TEST")
    print("A = swap red and blue")
    print("B = saturation scaling")

    # Apply A first, then B. No intermediate clipping.
    after_A = apply_filter(img, A, clip=False)
    sequential = apply_filter(after_A, B, clip=False)

    # Applying B @ A once must produce the same result.
    BA = B @ A
    combined = apply_filter(img, BA, clip=False)

    # Reverse order: A @ B
    AB = A @ B
    reversed_order = apply_filter(img, AB, clip=False)

    # Compare numerical results before clipping or saving.
    mse_match, max_match = error_between(sequential, combined)
    mse_reverse, max_reverse = error_between(
        sequential, reversed_order
    )

    print("\nProduct B @ A (A first, then B):")
    print(BA)

    print("\nProduct A @ B (B first, then A):")
    print(AB)

    print("\nSequential vs B @ A:")
    print(f"  Mean squared error = {mse_match:.12g}")
    print(f"  Maximum absolute error = {max_match:.12g}")

    print("\nSequential vs A @ B:")
    print(f"  Mean squared error = {mse_reverse:.12g}")
    print(f"  Maximum absolute error = {max_reverse:.12g}")

    assert np.allclose(sequential, combined), (
        "Composition test failed: sequential != B @ A"
    )

    assert not np.allclose(sequential, reversed_order), (
        "Choose matrices that do not commute."
    )

    print("\nPASS: Applying A then B equals applying B @ A.")
    print("PASS: Reversing the matrix order changes the result.")

    # Save all three composition images.
    save_image(
        sequential,
        f"{out_dir}/composition_A_then_B.png",
    )
    save_image(
        combined,
        f"{out_dir}/composition_BA_once.png",
    )
    save_image(
        reversed_order,
        f"{out_dir}/composition_AB_once.png",
    )

    # Side-by-side visual comparison.
    show_side_by_side(
        [sequential, combined, reversed_order],
        ["A then B", "B @ A once", "A @ B once"],
        save_to=f"{out_dir}/composition_comparison.png",
        show=False,  # FIX: plt.show() blocks main.py until the window is closed
    )

    print(f"\nComposition images saved in {out_dir}/")
    return results

In [ ]:
%%writefile inverse.py
"""
inverse.py -- PERSON 3
Matrix inverses, determinant/rank/condition analysis, image recovery,
a hand-coded 3x3 cofactor inverse, and inverse-of-product verification.

Keeps the required interface: run(img, out_dir, filtered=None)

Changes made during Person 1's review are marked with "# FIX".
"""

import os
import numpy as np
from utils import apply_filter, save_image, error_between, show_side_by_side
from filters import FILTERS


def inverse_3x3_by_hand(A, tol=1e-12):
    """Compute a 3x3 inverse using the determinant/cofactor/adjugate formula."""
    A = np.asarray(A, dtype=np.float64)
    if A.shape != (3, 3):
        raise ValueError(f"Expected a 3x3 matrix, got {A.shape}")

    a, b, c = A[0]
    d, e, f = A[1]
    g, h, i = A[2]

    # Cofactor matrix.
    C = np.array([
        [e * i - f * h, -(d * i - f * g), d * h - e * g],
        [-(b * i - c * h), a * i - c * g, -(a * h - b * g)],
        [b * f - c * e, -(a * f - c * d), a * e - b * d],
    ], dtype=np.float64)

    det = float(a * C[0, 0] + b * C[0, 1] + c * C[0, 2])
    if abs(det) <= tol:
        raise np.linalg.LinAlgError("Matrix is singular; inverse does not exist.")

    # A^-1 = transpose(cofactor matrix) / determinant.
    return C.T / det


def to_8bit(img):
    """Simulate saving an image as a normal 8-bit file and loading it back:
    clip to [0, 1], then round to the nearest of 256 levels."""
    return np.rint(np.clip(img, 0.0, 1.0) * 255.0) / 255.0


def run(img, out_dir, filtered=None):
    """Analyze filter matrices and test recovery/composition."""
    os.makedirs(out_dir, exist_ok=True)
    print("\nPERSON 3: MATRIX INVERSION AND IMAGE RECOVERY")

    print("\n1) Determinant, rank, condition number, and inverse")
    inverses = {}
    conds = {}
    for name, A in FILTERS.items():
        det = float(np.linalg.det(A))
        rank = int(np.linalg.matrix_rank(A))
        cond = float(np.linalg.cond(A))
        print(f"\n{name}:")
        print("  Matrix:\n", A)
        print(f"  determinant = {det:.6g}")
        print(f"  rank = {rank}")
        # FIX: a singular matrix has an infinite condition number; NumPy's
        # huge number (e.g. 6e49) is just rounding noise and confuses people.
        if rank < 3:
            print("  condition number = infinite (singular matrix)")
        else:
            print(f"  condition number = {cond:.6g}")

        if rank == 3 and np.isfinite(cond):
            inv_np = np.linalg.inv(A)
            inverses[name] = inv_np
            conds[name] = cond
            print("  inverse (NumPy):\n", np.array2string(inv_np, precision=6))
        else:
            print("  No inverse: this matrix is singular (information is lost).")

    # FIX: use the 'mix' matrix for the by-hand inverse. Its determinant is
    # 0.3025 and its entries are clean, so it can actually be worked out on
    # paper (and explained in the viva). Sepia's determinant is about 1.2e-7,
    # which is not realistic to do by hand.
    hand_name = "mix"
    hand_inv = inverse_3x3_by_hand(FILTERS[hand_name])
    np_inv = np.linalg.inv(FILTERS[hand_name])
    print(f"\n2) Hand-calculated cofactor inverse for '{hand_name}':")
    print(hand_inv)
    print("Hand inverse matches NumPy:",
          bool(np.allclose(hand_inv, np_inv, rtol=1e-8, atol=1e-8)))
    assert np.allclose(hand_inv, np_inv, rtol=1e-8, atol=1e-8)

    # ------------------------------------------------------------------
    # 3) Exact recovery: no clipping, no rounding. Shows the maths is right.
    # ------------------------------------------------------------------
    print("\n3) Image recovery, exact case (no clipping, no rounding)")
    for name, A in FILTERS.items():
        if name not in inverses:
            print(f"  {name}: skipped - singular matrix cannot recover original RGB.")
            continue
        transformed_raw = apply_filter(img, A, clip=False)
        recovered = apply_filter(transformed_raw, inverses[name], clip=False)
        mse, max_err = error_between(img, recovered)
        print(f"  {name}: MSE={mse:.3g}, max abs error={max_err:.3g}")
    print("  -> Every invertible filter is undone perfectly (errors are ~1e-12 or less).")

    # ------------------------------------------------------------------
    # 4) FIX: realistic recovery. The task is to apply A^-1 to the FILTERED
    #    images. Those are clipped to [0, 1] and saved as 8-bit files, which
    #    loses information. This is where conditioning shows up in practice.
    # ------------------------------------------------------------------
    print("\n4) Image recovery, realistic case (filtered image clipped to [0,1] "
          "and saved as 8-bit)")
    print(f"  {'filter':<11}{'cond':>11}{'MSE':>14}{'max abs err':>14}")
    recovery_errors = {}
    recovered_imgs = {}
    for name, A in FILTERS.items():
        if name not in inverses:
            continue
        if filtered is not None and name in filtered:
            saved = filtered[name]                 # already clipped by filters.run
        else:
            saved = apply_filter(img, A, clip=True)
        recovered = apply_filter(to_8bit(saved), inverses[name], clip=False)
        mse, max_err = error_between(img, recovered)
        recovery_errors[name] = (mse, max_err)
        recovered_imgs[name] = recovered
        print(f"  {name:<11}{conds[name]:>11.4g}{mse:>14.4g}{max_err:>14.4g}")
        save_image(recovered, os.path.join(out_dir, f"recovered_{name}.png"))

    print("  Why the errors differ:")
    print("   - swap_rb / identity only move numbers around, so nothing is lost.")
    print("   - saturate / brightness push some values above 1, which clipping")
    print("     cuts off; that information cannot come back.")
    print("   - sepia has a huge condition number, so even tiny rounding and")
    print("     clipping errors are blown up by A^-1: the recovery is garbage.")

    # Side-by-side figure for the report: original + every recovered image.
    names = [n for n in recovered_imgs if n != "identity"]
    show_side_by_side(
        [img] + [recovered_imgs[n] for n in names],
        ["original"] + [f"recovered ({n})" for n in names],
        save_to=os.path.join(out_dir, "recovery_comparison.png"),
        show=False,
    )

    # ------------------------------------------------------------------
    # 5) Singular case
    # ------------------------------------------------------------------
    gray = FILTERS.get("grayscale")
    if gray is not None:
        print("\n5) Singular grayscale case")
        print(f"  determinant = {np.linalg.det(gray):.6g}")
        print(f"  rank = {np.linalg.matrix_rank(gray)}")
        # FIX: a concrete example makes "information is lost" visible.
        c1 = np.array([0.9, 0.3, 0.3])   # a reddish colour
        c2 = np.array([0.3, 0.9, 0.3])   # a greenish colour
        print(f"  Two different colours: {c1} and {c2}")
        print(f"  grayscale({c1}) = {gray @ c1}")
        print(f"  grayscale({c2}) = {gray @ c2}")
        print("  Same output for different inputs, so no inverse can tell them apart.")

    # ------------------------------------------------------------------
    # 6) (B A)^-1 = A^-1 B^-1
    # ------------------------------------------------------------------
    print("\n6) Inverse of a product: (B @ A)^-1 = A^-1 @ B^-1")
    A = FILTERS["swap_rb"]
    B = FILTERS["saturate"]
    BA = B @ A
    left = np.linalg.inv(BA)
    right = np.linalg.inv(A) @ np.linalg.inv(B)
    print("(B @ A)^-1:\n", left)
    print("A^-1 @ B^-1:\n", right)
    print("Product inverse identity PASS:", bool(np.allclose(left, right)))
    assert np.allclose(left, right)

    print("\nPERSON 3 COMPLETE: inversion, recovery, singularity, and composition checks passed.")
    return {
        "inverses": inverses,
        "recovery_errors": recovery_errors,
        "hand_inverse": hand_inv,
    }

In [ ]:
%%writefile main.py
"""
main.py  --  PERSON 1: runs the whole project and saves everything to output/

Usage:   python main.py                       (uses images/photo.jpg)
         python main.py --image my_pic.jpg --out output
"""
import argparse
import os
import numpy as np

import utils
import filters
import inverse
from utils import banner


def stage1_load(path):
    banner("STAGE 1: Real-world data -> matrix representation",
           "An image is an H x W x 3 array; each pixel is an RGB vector",
           "Turn raw pixels into numbers a matrix can act on")
    img = utils.load_image(path)
    print("Loaded:", path)
    print("Image:", utils.image_info(img))
    pixels = img.reshape(-1, 3).T
    print(f"Pixel matrix P has shape {pixels.shape}  (3 rows = R,G,B ; one column per pixel)")
    print("First 3 pixel columns (RGB, 0-1):\n", np.round(pixels[:, :3], 3))
    return img


def stage2_selftest(img):
    banner("STAGE 2: Pipeline self-test (linear transformation A @ P)",
           "A 3x3 matrix acts on every pixel's RGB vector",
           "Prove the vectorised apply_filter() is correct before anyone relies on it")
    # Test 1: identity filter must return the image unchanged
    same = utils.apply_filter(img, np.eye(3))
    assert np.allclose(same, img), "Identity filter changed the image!"
    print("Identity matrix leaves the image unchanged ............ PASS")

    # Test 2: vectorised result == slow pixel-by-pixel loop
    A = filters.FILTERS["sepia"]
    crop = img[:20, :20]
    fast = utils.apply_filter(crop, A, clip=False)
    slow = np.zeros_like(crop)
    for i in range(crop.shape[0]):
        for j in range(crop.shape[1]):
            slow[i, j] = A @ crop[i, j]
    assert np.allclose(fast, slow)
    print("Vectorised filter == per-pixel loop (20x20 crop) ...... PASS")

    # Test 3: save/load round trip (only 8-bit rounding error allowed)
    utils.save_image(img, "output/_roundtrip.png")
    back = utils.load_image("output/_roundtrip.png")
    mse, mx = utils.error_between(img, back)
    assert mx <= 0.5 / 255 + 1e-9
    print(f"Save -> load round trip (max error {mx:.4f}) ........... PASS")
    os.remove("output/_roundtrip.png")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--image", default="images/photo.jpg")
    ap.add_argument("--out", default="output")
    args = ap.parse_args()

    if not os.path.exists(args.image):
        fallback = "images/sample.png"
        print(f"[!] {args.image} not found -> using synthetic {fallback}. "
              f"Put a REAL photo at {args.image} for the final run.")
        args.image = utils.make_sample_image(fallback)

    os.makedirs(args.out, exist_ok=True)
    img = stage1_load(args.image)
    utils.save_image(img, f"{args.out}/original.png")
    stage2_selftest(img)

    banner("STAGE 3: Filters and matrix multiplication (Person 2)",
           "Filter = 3x3 matrix; composition = matrix product",
           "Create colour effects; show B@A applies A then B")
    filtered = filters.run(img, args.out)

    banner("STAGE 4: Inverse, determinant, rank, conditioning (Person 3)",
           "A^-1 undoes A only if det(A) != 0 (rank 3)",
           "Recover originals; show which filters lose information")
    inverse.run(img, args.out, filtered)

    print(f"\nDone. All outputs are in '{args.out}/'.")


if __name__ == "__main__":
    main()

## 2. (Optional) Upload your real photo
Skip this cell to use the built-in synthetic test image.

In [ ]:
import os
os.makedirs('images', exist_ok=True)
try:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        os.replace(name, 'images/photo.jpg')
        print('Saved as images/photo.jpg')
        break
except ImportError:
    print('Not on Colab: put your photo at images/photo.jpg')

## 3. Run the full pipeline

In [ ]:
!python main.py

## 4. View results

In [ ]:
import glob, os
import utils
names = ['original'] + [os.path.basename(p)[len('filtered_'):-4]
         for p in sorted(glob.glob('output/filtered_*.png'))]
paths = ['output/original.png'] + sorted(glob.glob('output/filtered_*.png'))
imgs = [utils.load_image(p) for p in paths]
for k in range(0, len(imgs), 4):
    utils.show_side_by_side(imgs[k:k+4], names[k:k+4])